# Adult Income

## Câu 10. So sánh ba mô hình

### Yêu cầu

So sánh kết quả của cây entropy, CART và Gaussian Naïve Bayes trên cùng tập kiểm thử.

### Dữ liệu và phương pháp

Đọc trực tiếp ba bảng dự đoán đầy đủ ở câu 7–9; kiểm tra cùng mã mẫu và nhãn thực tế. So sánh accuracy và precision, recall, F1 macro trên cùng danh sách lớp. Macro là trung bình không trọng số giữa các lớp; không phải chỉ số riêng của một lớp. Chỉ số có mẫu số bằng 0 được quy ước bằng 0. Trường hợp bằng nhau được xác định bằng đúng giá trị đã tính, không đặt dung sai để gộp các kết quả gần nhau.

In [1]:
print("Đặng Ngọc Hiếu - 6451071024")

from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "datasets").is_dir():
    if (ROOT / "6451071024_Lab2" / "datasets").is_dir():
        ROOT = ROOT / "6451071024_Lab2"
        break
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Không tìm thấy thư mục datasets của LAB2.")
    ROOT = ROOT.parent
pd.set_option("display.max_columns", None)


Đặng Ngọc Hiếu - 6451071024


In [2]:
print("Đặng Ngọc Hiếu - 6451071024")

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
folder = ROOT / "Bai_2_Adult_Income/outputs"
tables = {}
reference = None
for number, model_name in [(7, "Cây entropy"), (8, "CART"), (9, "GaussianNB")]:
    path = folder / f"Cau{number:02}_du_doan.csv"
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu {path.name}; cần kết quả câu {number}.")
    table = pd.read_csv(path, dtype=str)
    required = {"sample_id", "thuc_te", "du_doan"}
    if not required.issubset(table.columns) or table.empty or table[list(required)].isna().any().any():
        raise ValueError(f"Bảng dự đoán {path.name} thiếu dữ liệu cần thiết.")
    if table["sample_id"].duplicated().any():
        raise ValueError("Mã mẫu bị trùng trong bảng dự đoán.")
    table = table.sort_values("sample_id").reset_index(drop=True)
    actual = table[["sample_id", "thuc_te"]]
    if reference is None:
        reference = actual
    elif not reference.equals(actual):
        raise ValueError("Ba mô hình không sử dụng cùng mẫu và nhãn kiểm thử.")
    tables[model_name] = table
# Dùng cùng miền nhãn cho mọi phép tính macro, kể cả lớp chỉ được một mô hình dự đoán.
classes = sorted(set(reference["thuc_te"]).union(
    *(set(table["du_doan"]) for table in tables.values())))
results = []
for model_name, table in tables.items():
    p, r, f, _ = precision_recall_fscore_support(
        table["thuc_te"], table["du_doan"], labels=classes,
        average="macro", zero_division=0)
    results.append({"Mô hình": model_name,
        "Accuracy": accuracy_score(table["thuc_te"], table["du_doan"]),
        "Precision macro": p, "Recall macro": r, "F1 macro": f})
comparison = pd.DataFrame(results).set_index("Mô hình")
display(comparison)
for metric in comparison.columns:
    best = comparison.index[comparison[metric].eq(comparison[metric].max())].tolist()
    print(f"{metric} cao nhất trên tập kiểm thử: {', '.join(best)} ({comparison[metric].max():.4f}).")
print("Các lớp dùng chung để tính macro:", classes)
print("Macro là trung bình không trọng số giữa cùng các lớp; các chỉ số có thể xếp hạng khác nhau.")
print("Kết luận chỉ áp dụng cho tập kiểm thử và phép chia dữ liệu này.")


Đặng Ngọc Hiếu - 6451071024


,Accuracy,Precision macro,Recall macro,F1 macro
Mô hình,,,,
Cây entropy,0.817530,0.753963,0.749749,0.751802
CART,0.812284,0.746791,0.741078,0.743831
GaussianNB,0.802922,0.749581,0.807313,0.764936


Accuracy cao nhất trên tập kiểm thử: Cây entropy (0.8175).
Precision macro cao nhất trên tập kiểm thử: Cây entropy (0.7540).
Recall macro cao nhất trên tập kiểm thử: GaussianNB (0.8073).
F1 macro cao nhất trên tập kiểm thử: GaussianNB (0.7649).
Các lớp dùng chung để tính macro: ['<=50K', '>50K']
Macro là trung bình không trọng số giữa cùng các lớp; các chỉ số có thể xếp hạng khác nhau.
Kết luận chỉ áp dụng cho tập kiểm thử và phép chia dữ liệu này.
